# Back-Translation Data Augmentation

This notebook expands the training data using back-translation, a technique that:
1. Uses an existing translation model to generate synthetic parallel pairs
2. Filters the synthetic pairs by quality using roundtrip translation
3. Mixes synthetic + real data at an optimal ratio

**Strategy:**
- Generate synthetic Twi from English monolingual data using En→Twi model
- Filter by roundtrip BLEU (Twi→En→compare with original)
- Mix ratio: 1 synthetic : 3 real (too much synthetic degrades quality)

**Use Case:** When you want to expand beyond the ~10K training pairs.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets sacrebleu sentencepiece accelerate pandas

## 2. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BACK-TRANSLATION CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

# Models for back-translation
EN_TO_TWI_MODEL = "ninte/en-twi-nllb-v2"    # For generating synthetic Twi
TWI_TO_EN_MODEL = "ninte/twi-en-nllb-v2"    # For roundtrip quality check

# Quality thresholds
ROUNDTRIP_THRESHOLD = 15.0   # Minimum sentence BLEU for roundtrip
MIX_RATIO = 3                # 1 synthetic : 3 real pairs

# Processing
MAX_LEN = 256
BATCH_SIZE = 32

# Output
OUTPUT_DIR = "/kaggle/working/augmented_data"

print("Configuration loaded.")
print(f"  En→Twi model: {EN_TO_TWI_MODEL}")
print(f"  Twi→En model: {TWI_TO_EN_MODEL}")
print(f"  Roundtrip threshold: {ROUNDTRIP_THRESHOLD}")
print(f"  Mix ratio: 1 synthetic : {MIX_RATIO} real")

## 3. Load Original Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

# Load datasets
print("Loading original datasets...")
ds1 = load_dataset("Ghana-NLP/TWI_ENGLISH_PARALLEL_TEXT")
ds2 = load_dataset("Ghana-NLP/ENGLISH_TWI_PARALLEL_TEXT")

df1 = ds1['train'].to_pandas().rename(columns={'text': 'twi', 'label': 'english'})
df2 = ds2['train'].to_pandas().rename(columns={'text': 'english', 'label': 'twi'})
df = pd.concat([df1[['twi', 'english']], df2[['twi', 'english']]], ignore_index=True)

# Clean
TWI_SPECIAL = set('ɛɔŋƐƆŊ')

def clean_text(text):
    if not isinstance(text, str) or pd.isna(text):
        return None
    text = re.sub(r'^\d+\.\s*', '', text.strip())
    text = re.sub(r' +', ' ', text)
    return text.strip() or None

df['twi'] = df['twi'].apply(clean_text)
df['english'] = df['english'].apply(clean_text)
df = df.dropna(subset=['twi', 'english'])

# Length filter
df['twi_wc'] = df['twi'].str.split().str.len()
df['eng_wc'] = df['english'].str.split().str.len()
df = df[(df['twi_wc'] >= 3) & (df['eng_wc'] >= 3)]
df = df[(df['twi_wc'] <= 150) & (df['eng_wc'] <= 150)]
df = df[df['twi'].apply(lambda x: not all(ord(c) < 128 for c in str(x).replace(' ', '')))]
df = df.drop_duplicates(subset=['twi', 'english'])
df = df.drop(columns=['twi_wc', 'eng_wc']).reset_index(drop=True)

# Mark as real data
df['synthetic'] = False

print(f"Cleaned real data: {len(df):,} pairs")

## 4. Extract English Monolingual Data

In [ ]:
# Get unique English sentences for back-translation
en_monolingual = df['english'].drop_duplicates().tolist()
print(f"English monolingual sentences: {len(en_monolingual):,}")

# Sample for demonstration (can remove this for full run)
# en_monolingual = en_monolingual[:1000]  # Uncomment to limit for testing

print(f"\nSample English sentences:")
for sent in en_monolingual[:3]:
    print(f"  - {sent[:80]}{'...' if len(sent) > 80 else ''}")

## 5. Load Translation Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load En→Twi model (for generating synthetic Twi)
print(f"\nLoading {EN_TO_TWI_MODEL}...")
en_twi_tokenizer = AutoTokenizer.from_pretrained(EN_TO_TWI_MODEL)
en_twi_model = AutoModelForSeq2SeqLM.from_pretrained(EN_TO_TWI_MODEL).to(device).eval()
print(f"  En→Twi model loaded.")

# Load Twi→En model (for roundtrip verification)
print(f"\nLoading {TWI_TO_EN_MODEL}...")
twi_en_tokenizer = AutoTokenizer.from_pretrained(TWI_TO_EN_MODEL)
twi_en_model = AutoModelForSeq2SeqLM.from_pretrained(TWI_TO_EN_MODEL).to(device).eval()
print(f"  Twi→En model loaded.")

## 6. Back-Translation: Generate Synthetic Twi

In [ ]:
from tqdm import tqdm

def translate_batch(texts, model, tokenizer, src_lang, tgt_lang, batch_size=32):
    """
    Translate a list of texts in batches.
    """
    translations = []
    
    # Get target token ID for NLLB
    tgt_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    
    for i in tqdm(range(0, len(texts), batch_size), desc=f"{src_lang}→{tgt_lang}"):
        batch = texts[i:i+batch_size]
        
        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=tgt_token_id,
                num_beams=4,
                max_length=MAX_LEN,
                no_repeat_ngram_size=3,
            )
        
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        translations.extend(decoded)
    
    return translations

In [ ]:
# Step 1: Generate synthetic Twi from English
print("Generating synthetic Twi...")
synthetic_twi = translate_batch(
    en_monolingual,
    en_twi_model,
    en_twi_tokenizer,
    src_lang="eng_Latn",
    tgt_lang="twi_Latn",
    batch_size=BATCH_SIZE
)

print(f"\nGenerated {len(synthetic_twi):,} synthetic Twi sentences.")
print("\nSamples:")
for en, twi in zip(en_monolingual[:3], synthetic_twi[:3]):
    print(f"  EN : {en[:60]}{'...' if len(en) > 60 else ''}")
    print(f"  TWI: {twi[:60]}{'...' if len(twi) > 60 else ''}")
    print()

## 7. Roundtrip Quality Filtering

In [ ]:
# Step 2: Translate synthetic Twi back to English
print("Roundtrip: Translating synthetic Twi back to English...")
roundtrip_english = translate_batch(
    synthetic_twi,
    twi_en_model,
    twi_en_tokenizer,
    src_lang="twi_Latn",
    tgt_lang="eng_Latn",
    batch_size=BATCH_SIZE
)

print(f"Generated {len(roundtrip_english):,} roundtrip translations.")

In [ ]:
import sacrebleu as sb

# Step 3: Score each pair by roundtrip BLEU
print(f"Filtering by roundtrip BLEU >= {ROUNDTRIP_THRESHOLD}...")

filtered_pairs = []
all_scores = []

for orig_en, syn_twi, rt_en in tqdm(zip(en_monolingual, synthetic_twi, roundtrip_english),
                                      total=len(en_monolingual),
                                      desc="Scoring"):
    # Calculate sentence-level BLEU between original and roundtrip English
    score = sb.sentence_bleu(rt_en, [orig_en]).score
    all_scores.append(score)
    
    if score >= ROUNDTRIP_THRESHOLD:
        filtered_pairs.append({
            'twi': syn_twi,
            'english': orig_en,
            'synthetic': True,
            'roundtrip_bleu': round(score, 2)
        })

print(f"\nSynthetic pairs passing threshold: {len(filtered_pairs):,} / {len(en_monolingual):,}")
print(f"Pass rate: {100*len(filtered_pairs)/len(en_monolingual):.1f}%")

In [ ]:
# Analyze roundtrip BLEU distribution
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_scores, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=ROUNDTRIP_THRESHOLD, color='red', linestyle='--', 
           label=f'Threshold: {ROUNDTRIP_THRESHOLD}')
ax.axvline(x=np.mean(all_scores), color='green', linestyle='--',
           label=f'Mean: {np.mean(all_scores):.1f}')
ax.set_xlabel('Roundtrip BLEU Score')
ax.set_ylabel('Frequency')
ax.set_title('Roundtrip BLEU Score Distribution')
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roundtrip_bleu_distribution.png', dpi=150)
plt.show()

print(f"\nRoundtrip BLEU Statistics:")
print(f"  Mean   : {np.mean(all_scores):.2f}")
print(f"  Median : {np.median(all_scores):.2f}")
print(f"  Std    : {np.std(all_scores):.2f}")

## 8. Mix Synthetic + Real Data

In [ ]:
# Create synthetic dataframe
synthetic_df = pd.DataFrame(filtered_pairs)

# Calculate mix amounts
n_synthetic = len(synthetic_df)
n_real_target = n_synthetic * MIX_RATIO
n_real_available = len(df)

print(f"Synthetic pairs: {n_synthetic:,}")
print(f"Target real pairs (1:{MIX_RATIO} ratio): {n_real_target:,}")
print(f"Available real pairs: {n_real_available:,}")

# Sample real data to match ratio (or use all if not enough)
if n_real_target <= n_real_available:
    real_sample = df.sample(n_real_target, random_state=42)
    print(f"Using {n_real_target:,} real pairs (sampled)")
else:
    real_sample = df.copy()
    print(f"Using all {n_real_available:,} real pairs")

# Ensure real data has same columns
real_sample = real_sample[['twi', 'english', 'synthetic']].copy()
synthetic_df = synthetic_df[['twi', 'english', 'synthetic']].copy()

In [ ]:
# Combine and shuffle
augmented_df = pd.concat([real_sample, synthetic_df], ignore_index=True)
augmented_df = augmented_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n{'='*60}")
print("AUGMENTED DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total pairs    : {len(augmented_df):,}")
print(f"Real pairs     : {(~augmented_df['synthetic']).sum():,}")
print(f"Synthetic pairs: {augmented_df['synthetic'].sum():,}")
print(f"Ratio          : 1 synthetic : {(~augmented_df['synthetic']).sum() / max(1, augmented_df['synthetic'].sum()):.1f} real")
print(f"{'='*60}")

## 9. Train/Val/Test Split (Augmented)

In [ ]:
from sklearn.model_selection import train_test_split

# IMPORTANT: Test and validation sets should be REAL DATA ONLY
# Never evaluate on synthetic data!

# Use original data for val/test
real_only = df.copy()
_, temp_df = train_test_split(real_only, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Use augmented data for training
train_df = augmented_df.copy()

print("Augmented splits:")
print(f"  Train: {len(train_df):,} (includes synthetic)")
print(f"  Val  : {len(val_df):,} (real only)")
print(f"  Test : {len(test_df):,} (real only)")

## 10. Save Augmented Dataset

In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save augmented dataset
augmented_df.to_csv(f'{OUTPUT_DIR}/train_augmented.csv', index=False)
val_df[['twi', 'english']].to_csv(f'{OUTPUT_DIR}/val_real.csv', index=False)
test_df[['twi', 'english']].to_csv(f'{OUTPUT_DIR}/test_real.csv', index=False)

# Save synthetic-only for analysis
synthetic_df.to_csv(f'{OUTPUT_DIR}/synthetic_pairs.csv', index=False)

print(f"\nSaved files to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f"  {f}: {size:.1f} KB")

## 11. Quality Check: Sample Synthetic Pairs

In [ ]:
# Show some high-quality synthetic pairs
print("\n" + "="*70)
print("HIGH-QUALITY SYNTHETIC PAIRS (Roundtrip BLEU > 30)")
print("="*70)

high_quality = [p for p in filtered_pairs if p['roundtrip_bleu'] > 30]
for pair in high_quality[:5]:
    print(f"\n  English: {pair['english'][:70]}{'...' if len(pair['english']) > 70 else ''}")
    print(f"  Twi    : {pair['twi'][:70]}{'...' if len(pair['twi']) > 70 else ''}")
    print(f"  RT BLEU: {pair['roundtrip_bleu']}")

print(f"\nTotal high-quality pairs (BLEU > 30): {len(high_quality):,}")

## 12. Summary

In [ ]:
print("\n" + "═"*70)
print("BACK-TRANSLATION SUMMARY")
print("═"*70)
print(f"\n  ── Input ──")
print(f"    English monolingual sentences: {len(en_monolingual):,}")
print(f"    Original parallel pairs: {len(df):,}")
print(f"\n  ── Generation ──")
print(f"    Synthetic Twi generated: {len(synthetic_twi):,}")
print(f"    Roundtrip threshold: {ROUNDTRIP_THRESHOLD}")
print(f"    Pairs passing threshold: {len(filtered_pairs):,}")
print(f"    Pass rate: {100*len(filtered_pairs)/len(en_monolingual):.1f}%")
print(f"\n  ── Augmented Dataset ──")
print(f"    Total training pairs: {len(augmented_df):,}")
print(f"    Real pairs: {(~augmented_df['synthetic']).sum():,}")
print(f"    Synthetic pairs: {augmented_df['synthetic'].sum():,}")
print(f"    Expansion: {100*len(augmented_df)/len(df):.0f}% of original")
print(f"\n  ── Output Files ──")
print(f"    {OUTPUT_DIR}/train_augmented.csv")
print(f"    {OUTPUT_DIR}/val_real.csv")
print(f"    {OUTPUT_DIR}/test_real.csv")
print("\n" + "═"*70)
print("\nNext step: Use train_augmented.csv for training.")
print("IMPORTANT: Always evaluate on real-only test set!")
print("═"*70)

## 13. Ablation Study Setup

To verify if back-translation helps, compare:
1. Train on real-only → Evaluate on test
2. Train on real+synthetic → Evaluate on test

If synthetic hurts performance, try:
- Increasing ROUNDTRIP_THRESHOLD
- Reducing MIX_RATIO
- Using domain-specific English data

In [ ]:
# Save real-only training set for ablation comparison
real_train, _ = train_test_split(df, test_size=0.2, random_state=42)
real_train[['twi', 'english']].to_csv(f'{OUTPUT_DIR}/train_real_only.csv', index=False)

print("Saved for ablation study:")
print(f"  {OUTPUT_DIR}/train_real_only.csv ({len(real_train):,} pairs)")
print(f"  {OUTPUT_DIR}/train_augmented.csv ({len(augmented_df):,} pairs)")
print("\nCompare training on each to determine if back-translation helps.")